In [ ]:
!pip install facenet-pytorch scikit-learn tqdm pillow streamlit opencv-python-headless pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os
real_csv_path = '/content/drive/MyDrive/faceforensics/csv/original.csv'
fake_csv_path = '/content/drive/MyDrive/faceforensics/csv/Deepfakes.csv'
base_video_dir = '/content/drive/MyDrive/faceforensics/'
real_df = pd.read_csv(real_csv_path)
fake_df = pd.read_csv(fake_csv_path)

In [ ]:
import cv2
import torch
from facenet_pytorch import MTCNN
from PIL import Image
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(keep_all=False, post_process=True, device=device)

def extract_faces_from_csv(df, base_dir, output_dir, max_videos=50, max_frames=10):
    os.makedirs(output_dir, exist_ok=True)
    processed_count = 0

    # Loop over video rows listed in your CSV file
    for index, row in tqdm(df.iterrows(), total=min(len(df), max_videos)):
        if processed_count >= max_videos:
            break

        video_rel_path = row['File Path'] # e.g., 'original/000.mp4'
        full_video_path = os.path.join(base_dir, video_rel_path)

        if not os.path.exists(full_video_path):
            continue

        cap = cv2.VideoCapture(full_video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        step = max(1, total_frames // max_frames)

        frame_idx = 0
        saved_frames = 0

        while cap.isOpened() and saved_frames < max_frames:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            if not ret:
                break

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(rgb_frame)
            boxes, _ = mtcnn.detect(pil_img)

            if boxes is not None:
                box = boxes[0].astype(int)
                box[0], box[1] = max(0, box[0]), max(0, box[1])
                cropped_face = pil_img.crop((box[0], box[1], box[2], box[3]))

                video_name = os.path.basename(video_rel_path).split('.')[0]
                cropped_face.save(os.path.join(output_dir, f"{video_name}_f{frame_idx}.jpg"))
                saved_frames += 1

            frame_idx += step
        cap.release()
        processed_count += 1
extract_faces_from_csv(real_df, base_video_dir, "dataset/real", max_videos=30)
extract_faces_from_csv(fake_df, base_video_dir, "dataset/fake", max_videos=30)


[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
100%|██████████| 30/30 [02:04<00:00,  4.13s/it]


In [ ]:
import os
import gc
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

gc.collect()
torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 16
EPOCHS = 8
LEARNING_RATE = 2e-5

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root='dataset', transform=train_transform)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

val_dataset.dataset.transform = val_transform

targets = [s[1] for s in full_dataset.samples]
class_counts = [targets.count(0), targets.count(1)]

class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
sample_weights = [class_weights[t] for t in targets]

train_indices = train_dataset.indices
train_sample_weights = [sample_weights[i] for i in train_indices]
sampler = WeightedRandomSampler(weights=train_sample_weights, num_samples=len(train_sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
model.heads.head = nn.Linear(model.heads.head.in_features, 2)

for param in model.parameters():
    param.requires_grad = True

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None

best_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}]")

    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        if scaler and device.type == 'cuda':
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            if device.type == 'cuda':
                with torch.cuda.amp.autocast():
                    outputs = model(images)
            else:
                outputs = model(images)

            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_acc = accuracy_score(all_labels, all_preds)

    if epoch_acc > best_acc:
        best_acc = epoch_acc
        os.makedirs("weights", exist_ok=True)
        checkpoint = {
            "state_dict": model.state_dict(),
            "class_to_idx": full_dataset.class_to_idx
        }
        torch.save(checkpoint, "weights/model_complete.pth")

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth
100%|██████████| 330M/330M [00:02<00:00, 147MB/s]
Epoch [8/8]: 100%|██████████| 30/30 [00:06<00:00,  4.39it/s]
